# ドライブレコーダーの 1 枚から、その続きを動画にする (Cosmos 3 Edge)

ドライブレコーダーの画像を 1 枚入れると、NVIDIA の世界モデル [nvidia/Cosmos3-Edge](https://huggingface.co/nvidia/Cosmos3-Edge) がその先の数秒を動画にします。

1. 画像をアップロードし、自分の車にどう動いてほしいかを選ぶ
2. Cosmos の **Reasoner** が画像を見て、道路・車・天気などを読み取り、生成用の JSON プロンプトを書く
3. Cosmos の **Generator** が、その JSON と画像から続きの動画を作る

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください。T4 では動きません。
上から順に実行すれば最後まで通ります。2 回目からは「1. 画像と動き」以降だけをやり直せば大丈夫です。

In [ ]:
# @title GPU を確かめる
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
major, _ = torch.cuda.get_device_capability()
assert major >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"

In [ ]:
# @title インストール (数分)
# Transformers は Cosmos3 Edge に対応したリリース、Diffusers は main ブランチが必要
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title プロンプトのお手本と、試し用の車載映像を取ってくる
from huggingface_hub import snapshot_download
repo = snapshot_download("nvidia/Cosmos3-Edge", allow_patterns=[
    "assets/example_i2v_prompt.json", "assets/negative_prompt.json", "assets/example_action_id_av_0_input.mp4"])

## 1. 画像と動き

- `use_sample` に印を付けると、アップロードの代わりにモデルカードのお手本の車載映像の最初のコマを使います (まず動くか試したいとき)
- `crop_bottom` は画像の下を切る割合です。日付・速度の表示やボンネットが映っているときに上げると、動画がきれいになります
- 画像は 16:9 (832×480) に合わせて中央を切り抜きます

In [ ]:
# @title 1. 画像と動き
import math
from google.colab import files
from PIL import Image as PILImage, ImageOps
from IPython.display import display

maneuver = "おまかせ (流れに乗って走り続ける)"  # @param ["おまかせ (流れに乗って走り続ける)", "まっすぐ進む", "減速して止まる", "左に曲がる", "右に曲がる"]
extra = ""  # @param {type:"string"}
# @markdown ↑ 付け足したいこと (日本語で OK)。例: 「前の車がブレーキを踏む」「信号が青に変わる」
num_frames = 61  # @param {type:"slider", min:53, max:149, step:4}
crop_bottom = 0.0  # @param {type:"slider", min:0, max:0.3, step:0.01}
use_sample = False  # @param {type:"boolean"}

MANEUVERS = {
    "おまかせ (流れに乗って走り続ける)": "The ego vehicle keeps driving naturally with the flow of traffic in its current lane.",
    "まっすぐ進む": "The ego vehicle continues straight ahead in its lane at a steady speed.",
    "減速して止まる": "The ego vehicle gradually slows down and comes to a smooth stop.",
    "左に曲がる": "The ego vehicle slows down and turns left at the next opportunity.",
    "右に曲がる": "The ego vehicle slows down and turns right at the next opportunity.",
}
FPS = 24
W, H = 832, 480

if use_sample:
    import av
    with av.open(f"{repo}/assets/example_action_id_av_0_input.mp4") as c:
        img = next(c.decode(video=0)).to_image()
else:
    up = files.upload()
    img = ImageOps.exif_transpose(PILImage.open(next(iter(up))))
img = img.convert("RGB")
if crop_bottom > 0:
    img = img.crop((0, 0, img.width, round(img.height * (1 - crop_bottom))))
first_frame = ImageOps.fit(img, (W, H), PILImage.BICUBIC)
first_frame.save("first_frame.png")
print(f"{num_frames} フレーム = {num_frames / FPS:.1f} 秒の続きを作ります")
display(first_frame.resize((W // 2, H // 2)))

## 2. Reasoner に場面を読ませて、JSON プロンプトを書かせる

Generator は短い文より、場面を細かく書いた決まった形の JSON のほうがよく動きます。それを Cosmos の Reasoner に書かせます (API キー不要)。
GPU メモリを空けるため、書き終わったら Reasoner は降ろします。

In [ ]:
# @title 2. JSON プロンプトを書かせる
import gc, json, torch
from pathlib import Path
from transformers import AutoModelForImageTextToText, AutoProcessor

for name in ("pipe", "model", "processor"):  # 前に載せたモデルを GPU から降ろす
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

example = json.loads(Path(f"{repo}/assets/example_i2v_prompt.json").read_text())
duration = num_frames / FPS
request = f"""You write prompts for an image-to-video world model.
The attached image is a frame from a forward-facing dashcam mounted behind the windshield of a car (the ego vehicle).
The video starts exactly from this frame and continues the drive realistically for {duration:.1f} seconds.
What the ego vehicle does: {MANEUVERS[maneuver]}
Extra details (may be written in Japanese; translate them): {extra or "none"}

Describe this video as one JSON object with exactly the same keys as the example below.
First describe what is really in the image: road layout, lanes and markings, other vehicles and pedestrians,
signs and traffic lights, buildings, weather, time of day and lighting.
Then describe how the scene changes from the dashcam's point of view, with timestamps in "actions" and "segments":
the camera moves with the ego vehicle, the road and the surroundings move toward and past the camera,
and other road users keep moving in physically plausible ways.
The camera is fixed to the car: no cuts, no zoom, no orbiting.
Use "" for keys that do not apply. Write every value in English. Output only the JSON.

Example:
{json.dumps(example, ensure_ascii=False)}"""

messages = [{"role": "user", "content": [
    {"type": "image", "path": "first_frame.png"},
    {"type": "text", "text": request},
]}]
inputs = processor.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_dict=True,
    return_tensors="pt", enable_thinking=False,
).to(model.device)
out = model.generate(**inputs, max_new_tokens=3072)
text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

try:
    json_prompt = json.loads(text[text.find("{"): text.rfind("}") + 1])
except json.JSONDecodeError:
    # 崩れた JSON が返ってきたら、動きの説明だけを入れた最小の形で代用する
    print("JSON として読めなかったので、動きの説明だけで組みます\n", text[:500])
    caption = f"Forward-facing dashcam view from a moving car. {MANEUVERS[maneuver]} {extra}".strip()
    json_prompt = {k: ([] if isinstance(v, list) else {} if isinstance(v, dict) else "") for k, v in example.items()}
    json_prompt["temporal_caption"] = caption
    json_prompt["actions"] = [{"time": f"0:00-0:{math.ceil(duration):02d}", "description": caption}]

# 解像度と長さは実際に作る値に揃える
json_prompt.update({"resolution": {"W": W, "H": H}, "aspect_ratio": "16,9",
                    "duration": f"{round(duration)}s", "fps": FPS})
Path("dashcam_prompt.json").write_text(json.dumps(json_prompt, indent=2, ensure_ascii=False))
print(json.dumps(json_prompt, indent=2, ensure_ascii=False)[:3000])

del model, processor
gc.collect(); torch.cuda.empty_cache()

JSON を手で直したいときは、左のファイル欄で `dashcam_prompt.json` をダブルクリックして編集・保存してから次のセルを実行してください (次のセルはこのファイルを読み直します)。

## 3. 続きの動画を作る

目安: 61 フレームで A100 なら数分、L4 ではその数倍かかります。初回はモデルの読み込みにも時間がかかります。
`n_videos` を増やすと、seed を 1 つずつずらして何本か作ります。同じ写真でも seed で出来がかなり変わるので、何本か作って選ぶのがおすすめです。

In [ ]:
# @title 3. 動画を作る
import json, torch
from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Video

seed = 0  # @param {type:"integer"}
n_videos = 1  # @param {type:"slider", min:1, max:4, step:1}
download = False  # @param {type:"boolean"}

if "pipe" not in globals():
    pipe = Cosmos3OmniPipeline.from_pretrained("nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=True)
    if torch.cuda.get_device_properties(0).total_memory / 1e9 < 40:
        pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
    else:
        pipe.to("cuda")
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=12.0, use_karras_sigmas=False)

json_prompt = json.loads(Path("dashcam_prompt.json").read_text())
negative_prompt = json.loads(Path(f"{repo}/assets/negative_prompt.json").read_text())

for s in range(seed, seed + n_videos):
    result = pipe(
        prompt=json.dumps(json_prompt),
        negative_prompt=json.dumps(negative_prompt),
        image=PILImage.open("first_frame.png"),
        num_frames=num_frames,
        height=H,
        width=W,
        fps=float(FPS),
        num_inference_steps=20,
        guidance_scale=6.0,
        generator=torch.Generator(device="cuda").manual_seed(s),
        add_resolution_template=False,
        add_duration_template=False,
    )
    out_path = f"dashcam_seed{s}.mp4"
    export_to_video(result.video, out_path, fps=FPS, macro_block_size=1)
    print(out_path)
    display(Video(out_path, embed=True, width=W))
    if download:
        files.download(out_path)

## うまくいかないとき

- **車がほとんど進まない**: 2 の JSON の `actions` と `segments` に「道路がカメラに向かって流れてくる」ような動きを具体的に書き足すと効きます
- **景色が崩れる・別の場所になる**: 長い動画ほど崩れやすいので、`num_frames` を減らすか、seed を変えて何本か作ってください
- **日付や速度の文字がにじむ**: 1 の `crop_bottom` で切り落としてください
- **真っ黒・空の動画になる**: 安全フィルタ (`cosmos_guardrail`) に止められた可能性があります。seed や画像を変えて試してください
- 作られるのは「それらしい続き」で、実際に起きたことの再現ではありません